In [2]:
import matplotlib
# Use the PGF backend, which is designed for LaTeX integration.
# This MUST be called before importing pyplot.
matplotlib.use('pgf')
import numpy as np
import scipy.integrate
import matplotlib.pyplot as plt
import os

# latex stuff
# Configure matplotlib to use LaTeX for all text rendering.
plt.rcParams.update({
    "font.family": "serif",  # Use the document's default serif font
    "text.usetex": True,     # Enable LaTeX for text processing
    "pgf.rcfonts": False,    # Let LaTeX manage the fonts, not matplotlib
    "pgf.preamble": "\n".join([
        r"\usepackage[utf8x]{inputenc}",
        r"\usepackage[T1]{fontenc}",
        r"\usepackage{amsmath}", # For math symbols
    ]),
})

# params
# Model parameters
E_PARAM = 0.25  # Epsilon
D_PARAM = 0.5   # Delta

# Simulation parameters
MATRIX_SIZE = 10
P_VALS = np.linspace(0, 1, 11)
Q_VALS = np.linspace(0, 1, 11)
NUM_TRIALS = 50  # Number of runs for each (p,q) pair

# ODE solver and convergence check parameters
TIME_SPAN = [0, 600]
CONVERGENCE_WINDOW = 25  # How many recent time steps to check for convergence
CONVERGENCE_TOL = 0.01   # Tolerance for checking if values are stable

# optimized functions

def generate_weight_matrix(matrix_size, p, q):
    """
    Generates a weight matrix using a fully vectorized approach for efficiency.
    """
    # Probabilities for choosing one of the four interaction types
    weights = [1 - p, p * (1 - q) / 2, p * (1 - q) / 2, p * q]
    possible_vals = np.array([-2, -1, 1, 2])

    # Get indices for the upper triangle of the matrix (where i < j)
    i_upper, j_upper = np.triu_indices(matrix_size, k=1)
    n_upper = len(i_upper)

    # Make all random choices for the upper triangle in one vectorized call
    choices = np.random.choice(possible_vals, size=n_upper, p=weights)

    # Pre-calculate weight values from parameters
    w_excited = -1 + E_PARAM
    w_inhibited = -1 - D_PARAM 

    # Initialize the final weight matrix
    mat = np.zeros((matrix_size, matrix_size))
    
    # Create arrays to hold the values for W_ij and W_ji
    vals_ij = np.zeros(n_upper)
    vals_ji = np.zeros(n_upper)

    # Apply rules using boolean masks for vectorization
    mask_2 = (choices == 2)
    vals_ij[mask_2] = w_excited
    vals_ji[mask_2] = w_excited

    mask_neg2 = (choices == -2)
    vals_ij[mask_neg2] = w_inhibited
    vals_ji[mask_neg2] = w_inhibited

    mask_neg1 = (choices == -1)
    vals_ij[mask_neg1] = w_inhibited
    vals_ji[mask_neg1] = w_excited

    mask_1 = (choices == 1)
    vals_ij[mask_1] = w_excited
    vals_ji[mask_1] = w_inhibited

    # Assign the calculated values to the matrix's upper and lower triangles
    mat[i_upper, j_upper] = vals_ij
    mat[j_upper, i_upper] = vals_ji

    return mat

def system_ode(t, x, W):
    """The system of differential equations to be solved."""
    return -x + np.maximum(0, W @ x + 1)

# main script

# --- Setup ---
num_p = len(P_VALS)
num_q = len(Q_VALS)
convergence_heatmap = np.zeros((num_q, num_p))

# Create a figure for the time-series trajectories
# NOTE: figsize is in INCHES.
fig_trajectories, axs = plt.subplots(num_q, num_p, figsize=(15, 15), 
                                     sharex=True, sharey=True,
                                     constrained_layout=True)
# Use LaTeX commands for font sizing
fig_trajectories.suptitle(r'\Huge System Trajectories for Matrix Size ' + f'{MATRIX_SIZE}')

print("Starting simulation...")


# Storage
all_final_states = {}  # Key: (matrix_size, p_idx, q_idx), Value: list of final states
all_weight_matrices = {}  # Key: (matrix_size, p_idx, q_idx), Value: list of weight matrices
all_recent_trajectories = {}
current_matrix_size = MATRIX_SIZE

# --- Simulation Loop ---
for p_idx, p in enumerate(P_VALS):
    for q_idx, q in enumerate(Q_VALS):
        trial_convergence_results = []
        
        # Run several trials for statistical robustness
        for trial_num in range(NUM_TRIALS):
            W = generate_weight_matrix(MATRIX_SIZE, p, q)
            x0 = np.random.rand(MATRIX_SIZE) # Random initial conditions

            # Solve the ODE
            sol = scipy.integrate.solve_ivp(
                lambda t, x: system_ode(t, x, W), 
                TIME_SPAN, 
                x0
            )

            # Store final states for later analysis
            key = (current_matrix_size, p_idx, q_idx)
            if key not in all_final_states:
                all_final_states[key] = []
                all_weight_matrices[key] = []
                all_recent_trajectories[key] = []
            all_final_states[key].append(sol.y[:, -1].copy())
            all_weight_matrices[key].append(W.copy())  # Store the weight matrix too
            all_recent_trajectories[key].append(sol.y[:, -15:].copy())

            # Vectorized Convergence Check
            is_converged = False
            if sol.y.shape[1] >= CONVERGENCE_WINDOW:
                final_values = sol.y[:, -1, np.newaxis]
                recent_points = sol.y[:, -CONVERGENCE_WINDOW:]
                if np.all(np.abs(final_values - recent_points) <= CONVERGENCE_TOL):
                    is_converged = True
            
            trial_convergence_results.append(1 if is_converged else 0)

        # Store the average convergence rate for this (p, q) pair
        convergence_heatmap[q_idx, p_idx] = np.mean(trial_convergence_results)
        print(f"p={p:.1f}, q={q:.1f} | Mean Convergence: {np.mean(trial_convergence_results):.2f}")

        # Plot trajectories on the correct subplot
        ax = axs[num_q - 1 - q_idx, p_idx]
        ax.plot(sol.t, sol.y.T, lw=0.5) # Made lines thinner
        # Use LaTeX formatting for titles and labels
        ax.set_title(fr'$p={p:.1f}$, $q={q:.1f}$', fontsize=8)
        
        # Add labels only to the outer plots
        if q_idx == 0: # Bottom row
             ax.set_xlabel(r'Time $t$')
        if p_idx == 0: # Leftmost column
             ax.set_ylabel(r'State $x_i$')
        
        # Set tick font size (in points)
        ax.tick_params(axis='both', which='major', labelsize=6)

print("\nSimulation complete. Generating heatmap...")

Starting simulation...
p=0.0, q=0.0 | Mean Convergence: 1.00
p=0.0, q=0.1 | Mean Convergence: 1.00
p=0.0, q=0.2 | Mean Convergence: 1.00
p=0.0, q=0.3 | Mean Convergence: 1.00
p=0.0, q=0.4 | Mean Convergence: 1.00
p=0.0, q=0.5 | Mean Convergence: 1.00
p=0.0, q=0.6 | Mean Convergence: 1.00
p=0.0, q=0.7 | Mean Convergence: 1.00
p=0.0, q=0.8 | Mean Convergence: 1.00
p=0.0, q=0.9 | Mean Convergence: 1.00
p=0.0, q=1.0 | Mean Convergence: 1.00
p=0.1, q=0.0 | Mean Convergence: 0.98
p=0.1, q=0.1 | Mean Convergence: 0.96
p=0.1, q=0.2 | Mean Convergence: 1.00
p=0.1, q=0.3 | Mean Convergence: 0.98
p=0.1, q=0.4 | Mean Convergence: 1.00
p=0.1, q=0.5 | Mean Convergence: 1.00
p=0.1, q=0.6 | Mean Convergence: 1.00
p=0.1, q=0.7 | Mean Convergence: 1.00
p=0.1, q=0.8 | Mean Convergence: 1.00
p=0.1, q=0.9 | Mean Convergence: 1.00
p=0.1, q=1.0 | Mean Convergence: 1.00
p=0.2, q=0.0 | Mean Convergence: 0.90
p=0.2, q=0.1 | Mean Convergence: 0.98
p=0.2, q=0.2 | Mean Convergence: 0.96
p=0.2, q=0.3 | Mean Converg

In [5]:
# --- Figure: Mean Convergence Heatmap ---

# 1. Calculate the step size between points
p_step = P_VALS[1] - P_VALS[0] if len(P_VALS) > 1 else 1
q_step = Q_VALS[1] - Q_VALS[0] if len(Q_VALS) > 1 else 1

# 2. Pad the extent by half a step so pixels are perfectly centered on the values
extent = [
    P_VALS[0] - p_step/2, P_VALS[-1] + p_step/2, 
    Q_VALS[0] - q_step/2, Q_VALS[-1] + q_step/2
]

# Create figure
fig_conv, ax_conv = plt.subplots(figsize=(8, 7), constrained_layout=True)

# 3. Plot the heatmap using the padded extent
im_conv = ax_conv.imshow(
    convergence_heatmap, 
    origin='lower',
    extent=extent,
    aspect='auto',
    cmap='RdYlBu', # Matches your old script's color scheme
    vmin=0, 
    vmax=1
)

# Colorbar
cbar_conv = fig_conv.colorbar(im_conv, ax=ax_conv)
cbar_conv.set_label(r'Mean Convergence Probability', fontsize=11)

# ---------------------------------------------------------
# NEW: Overlay the function p = 4q / (1+q)^2
# ---------------------------------------------------------
# Generate dense values for a smooth curve
q_curve = np.linspace(0, 1, 100)
p_curve = (4 * q_curve) / (1 + q_curve)**2

# Plot the curve over the heatmap
ax_conv.plot(
    p_curve, 
    q_curve, 
    color='white',       # Contrast against the heatmap
    linewidth=2.5, 
    linestyle='--', 
    label=r'$p = \frac{4q}{(1+q)^2}$'
)

# Add a legend for the overlaid function
ax_conv.legend(loc='lower right', fontsize=11, framealpha=0.9)
# ---------------------------------------------------------

# 4. Explicitly map the ticks to your exact P and Q arrays
ax_conv.set_xticks(P_VALS)
ax_conv.set_yticks(Q_VALS)

# Labels and formatting
ax_conv.set_xlabel(r'$p$ (interaction probability)', fontsize=11)
ax_conv.set_ylabel(r'$q$ (excitatory probability)', fontsize=11)
ax_conv.set_title(r'\Large Convergence Heatmap (N=' + f'{MATRIX_SIZE})', fontsize=14, pad=15)

# Optional: Add gridlines to clearly separate the bins
ax_conv.grid(True, alpha=0.2, linestyle=':', linewidth=0.5)

# Save the figure to your analysis folder
filepath_conv_pdf = os.path.join(run_folder, 'convergence_heatmap.pdf')
filepath_conv_pgf = os.path.join(run_folder, 'convergence_heatmap.pgf')
fig_conv.savefig(filepath_conv_pdf, bbox_inches='tight')
fig_conv.savefig(filepath_conv_pgf, bbox_inches='tight')

# plt.show() # Uncomment if you are viewing this inline

In [4]:
# folder creation
import os
from datetime import datetime

# Specify which matrix size to analyze
ANALYZE_MATRIX_SIZE = 10  # Change this as needed

# Create a timestamped folder for this analysis run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"analysis_run_{timestamp}_N{ANALYZE_MATRIX_SIZE}"
os.makedirs(run_folder, exist_ok=True)

print(f"Created analysis folder: {run_folder}")
print(f"All figures will be saved to this folder.\n")



Created analysis folder: analysis_run_20260408_141409_N10
All figures will be saved to this folder.



In [4]:
# block 1 active
print("="*70)
print("BLOCK 1: ACTIVE NEURON ANALYSIS")
print("="*70)

ACTIVE_THRESHOLD = 0.05

active_mean_heatmap = np.zeros((num_q, num_p))
active_std_heatmap = np.zeros((num_q, num_p))
active_min_heatmap = np.zeros((num_q, num_p))
active_max_heatmap = np.zeros((num_q, num_p))

print(f"Analyzing active neurons (threshold: x > {ACTIVE_THRESHOLD})\n")

for p_idx in range(num_p):
    for q_idx in range(num_q):
        key = (ANALYZE_MATRIX_SIZE, p_idx, q_idx)
        
        if key not in all_final_states:
            print(f"Warning: No data for p_idx={p_idx}, q_idx={q_idx}")
            continue
        
        trial_active_counts = []
        for final_state in all_final_states[key]:
            num_active = np.sum(final_state > ACTIVE_THRESHOLD)
            trial_active_counts.append(num_active)
        
        active_mean_heatmap[q_idx, p_idx] = np.mean(trial_active_counts)
        active_std_heatmap[q_idx, p_idx] = np.std(trial_active_counts)
        active_min_heatmap[q_idx, p_idx] = np.min(trial_active_counts)
        active_max_heatmap[q_idx, p_idx] = np.max(trial_active_counts)
        
        p_val = P_VALS[p_idx]
        q_val = Q_VALS[q_idx]
        print(f"p={p_val:.1f}, q={q_val:.1f} | Active: {np.mean(trial_active_counts):.1f}±{np.std(trial_active_counts):.1f} "
              f"[{np.min(trial_active_counts):.0f}, {np.max(trial_active_counts):.0f}]")

# viz active

p_step = P_VALS[1] - P_VALS[0] if len(P_VALS) > 1 else 1
q_step = Q_VALS[1] - Q_VALS[0] if len(Q_VALS) > 1 else 1
extent = [P_VALS[0] - p_step/2, P_VALS[-1] + p_step/2, 
          Q_VALS[0] - q_step/2, Q_VALS[-1] + q_step/2]

# Adaptive color scale based on logarithmic scaling with N
# Max active neurons ~ c * log(N), empirically calibrated
expected_max_active = max(ANALYZE_MATRIX_SIZE * 0.8, 3 * np.log(ANALYZE_MATRIX_SIZE))

# expected_max_active = 1.5 * np.log(ANALYZE_MATRIX_SIZE)
actual_max = np.max(active_mean_heatmap)
vmax_count = min(ANALYZE_MATRIX_SIZE, max(expected_max_active, actual_max * 1.1))
vmax_std = vmax_count / 4

print(f"\nColor scale info:")
print(f"  Expected max active (log scale): {expected_max_active:.1f}")
print(f"  Actual max observed: {actual_max:.1f}")
print(f"  Using vmax: {vmax_count:.1f}")

# --- Figure 1: Four-panel statistics ---
fig_active_stats, axes_stats = plt.subplots(2, 2, figsize=(12, 12.5), constrained_layout=True)
# Add extra space at the top for the title
fig_active_stats.suptitle(r'\Large Active Neuron Statistics (N=' + f'{ANALYZE_MATRIX_SIZE})', 
                          fontsize=14, y=0.985)

# Mean
im1 = axes_stats[0, 0].imshow(active_mean_heatmap, origin='lower', 
                               extent=extent, aspect='auto', 
                               cmap='viridis', interpolation='nearest',
                               vmin=0, vmax=vmax_count)
axes_stats[0, 0].set_title(r'Mean Active Neurons', fontsize=11)
axes_stats[0, 0].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_stats[0, 0].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_stats[0, 0].set_xticks(P_VALS)
axes_stats[0, 0].set_yticks(Q_VALS)
axes_stats[0, 0].tick_params(labelsize=9)
cbar1 = fig_active_stats.colorbar(im1, ax=axes_stats[0, 0])
cbar1.set_label(r'Mean count', fontsize=10)
contours1 = axes_stats[0, 0].contour(P_VALS, Q_VALS, active_mean_heatmap,
                                      colors='white', alpha=0.3, linewidths=0.5, levels=5)
axes_stats[0, 0].clabel(contours1, inline=True, fontsize=6)

# Std Dev
im2 = axes_stats[0, 1].imshow(active_std_heatmap, origin='lower', 
                               extent=extent, aspect='auto', 
                               cmap='plasma', interpolation='nearest',
                               vmin=0, vmax=vmax_std)
axes_stats[0, 1].set_title(r'Std Dev of Active Neurons', fontsize=11)
axes_stats[0, 1].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_stats[0, 1].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_stats[0, 1].set_xticks(P_VALS)
axes_stats[0, 1].set_yticks(Q_VALS)
axes_stats[0, 1].tick_params(labelsize=9)
cbar2 = fig_active_stats.colorbar(im2, ax=axes_stats[0, 1])
cbar2.set_label(r'Std dev', fontsize=10)

# Min
im3 = axes_stats[1, 0].imshow(active_min_heatmap, origin='lower', 
                               extent=extent, aspect='auto', 
                               cmap='cividis', interpolation='nearest',
                               vmin=0, vmax=vmax_count)
axes_stats[1, 0].set_title(r'Minimum Active Neurons', fontsize=11)
axes_stats[1, 0].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_stats[1, 0].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_stats[1, 0].set_xticks(P_VALS)
axes_stats[1, 0].set_yticks(Q_VALS)
axes_stats[1, 0].tick_params(labelsize=9)
cbar3 = fig_active_stats.colorbar(im3, ax=axes_stats[1, 0])
cbar3.set_label(r'Min count', fontsize=10)

# Max
im4 = axes_stats[1, 1].imshow(active_max_heatmap, origin='lower', 
                               extent=extent, aspect='auto', 
                               cmap='inferno', interpolation='nearest',
                               vmin=0, vmax=vmax_count)
axes_stats[1, 1].set_title(r'Maximum Active Neurons', fontsize=11)
axes_stats[1, 1].set_xlabel(r'$p$ (interaction probability)', fontsize=10)
axes_stats[1, 1].set_ylabel(r'$q$ (excitatory probability)', fontsize=10)
axes_stats[1, 1].set_xticks(P_VALS)
axes_stats[1, 1].set_yticks(Q_VALS)
axes_stats[1, 1].tick_params(labelsize=9)
cbar4 = fig_active_stats.colorbar(im4, ax=axes_stats[1, 1])
cbar4.set_label(r'Max count', fontsize=10)

filepath_stats_pdf = os.path.join(run_folder, 'active_neurons_statistics.pdf')
filepath_stats_pgf = os.path.join(run_folder, 'active_neurons_statistics.pgf')
fig_active_stats.savefig(filepath_stats_pdf, bbox_inches='tight')
fig_active_stats.savefig(filepath_stats_pgf, bbox_inches='tight')

# --- Figure 2: Fraction of active neurons ---
active_fraction_heatmap = active_mean_heatmap / ANALYZE_MATRIX_SIZE

fig_active_frac, ax_frac = plt.subplots(1, 1, figsize=(8, 7), constrained_layout=True)
im_frac = ax_frac.imshow(active_fraction_heatmap, origin='lower',
                          extent=extent, aspect='auto',
                          cmap='RdYlGn', interpolation='nearest',
                          vmin=0, vmax=1)
ax_frac.set_title(r'\Large Fraction of Active Neurons (N=' + f'{ANALYZE_MATRIX_SIZE})', 
                  fontsize=14, pad=20)
ax_frac.set_xlabel(r'$p$ (interaction probability)', fontsize=11)
ax_frac.set_ylabel(r'$q$ (excitatory probability)', fontsize=11)
ax_frac.set_xticks(P_VALS)
ax_frac.set_yticks(Q_VALS)
ax_frac.tick_params(labelsize=9)

cbar_frac = fig_active_frac.colorbar(im_frac, ax=ax_frac)
cbar_frac.set_label(r'Fraction active', fontsize=11)

contours_frac = ax_frac.contour(P_VALS, Q_VALS, active_fraction_heatmap,
                                 levels=np.linspace(0, 1, 11),
                                 colors='black', alpha=0.4, linewidths=0.8)
ax_frac.clabel(contours_frac, inline=True, fontsize=7, fmt='%.1f')
ax_frac.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)

filepath_frac_pdf = os.path.join(run_folder, 'active_neurons_fraction.pdf')
filepath_frac_pgf = os.path.join(run_folder, 'active_neurons_fraction.pgf')
fig_active_frac.savefig(filepath_frac_pdf, bbox_inches='tight')
fig_active_frac.savefig(filepath_frac_pgf, bbox_inches='tight')

print("\n" + "="*70)
print("BLOCK 1 SUMMARY: ACTIVE NEURONS")
print("="*70)
print(f"Threshold: x > {ACTIVE_THRESHOLD}")
print(f"Mean active neurons: {np.mean(active_mean_heatmap):.2f} ± {np.std(active_mean_heatmap):.2f}")
print(f"Mean fraction active: {np.mean(active_fraction_heatmap):.3f} ± {np.std(active_fraction_heatmap):.3f}")
print(f"Range: [{np.min(active_mean_heatmap):.1f}, {np.max(active_mean_heatmap):.1f}] neurons")
print(f"Max variability (std): {np.max(active_std_heatmap):.2f}")
print(f"\nFigures saved to: {run_folder}/")
print(f"  - active_neurons_statistics.pdf/pgf")
print(f"  - active_neurons_fraction.pdf/pgf")
print("="*70 + "\n")

BLOCK 1: ACTIVE NEURON ANALYSIS
Analyzing active neurons (threshold: x > 0.05)

p=0.0, q=0.0 | Active: 1.0±0.0 [1, 1]
p=0.0, q=0.1 | Active: 1.0±0.0 [1, 1]
p=0.0, q=0.2 | Active: 1.0±0.0 [1, 1]
p=0.0, q=0.3 | Active: 1.0±0.0 [1, 1]
p=0.0, q=0.4 | Active: 1.0±0.0 [1, 1]
p=0.0, q=0.5 | Active: 1.0±0.0 [1, 1]
p=0.0, q=0.6 | Active: 1.0±0.0 [1, 1]
p=0.0, q=0.7 | Active: 1.0±0.0 [1, 1]
p=0.0, q=0.8 | Active: 1.0±0.0 [1, 1]
p=0.0, q=0.9 | Active: 1.0±0.0 [1, 1]
p=0.0, q=1.0 | Active: 1.0±0.0 [1, 1]
p=0.1, q=0.0 | Active: 1.0±0.0 [1, 1]
p=0.1, q=0.1 | Active: 1.2±0.4 [1, 2]
p=0.1, q=0.2 | Active: 1.5±0.5 [1, 2]
p=0.1, q=0.3 | Active: 1.6±0.5 [1, 2]
p=0.1, q=0.4 | Active: 1.6±0.5 [1, 2]
p=0.1, q=0.5 | Active: 1.9±0.3 [1, 2]
p=0.1, q=0.6 | Active: 1.7±0.5 [1, 2]
p=0.1, q=0.7 | Active: 1.9±0.3 [1, 2]
p=0.1, q=0.8 | Active: 2.0±0.0 [2, 2]
p=0.1, q=0.9 | Active: 2.1±0.3 [2, 3]
p=0.1, q=1.0 | Active: 2.1±0.3 [2, 3]
p=0.2, q=0.0 | Active: 1.1±0.3 [1, 2]
p=0.2, q=0.1 | Active: 1.6±0.9 [1, 4]
p=0.2, q

In [6]:
print("="*70)
print("BLOCK 2: DYNAMICAL REGIME CLASSIFICATION (FIXED POINT VS NONLINEAR)")
print("="*70)

ACTIVE_THRESHOLD = 0.05
DERIV_ERROR_THRESHOLD = 0.01

# Initialize heatmaps with NaNs so empty regions remain blank
fp_mean_heatmap = np.full((num_q, num_p), np.nan)
nl_mean_heatmap = np.full((num_q, num_p), np.nan)

print("Classifying network realizations...")
print(f"  Active threshold: x > {ACTIVE_THRESHOLD}")
print(f"  Fixed point threshold: max(dx) - min(dx) < {DERIV_ERROR_THRESHOLD} over last 15 steps\n")

for p_idx in range(num_p):
    for q_idx in range(num_q):
        key = (ANALYZE_MATRIX_SIZE, p_idx, q_idx)
        
        if key not in all_recent_trajectories or key not in all_weight_matrices:
            continue
            
        trial_fp_counts = []
        trial_nl_counts = []
        
        for recent_x, W in zip(all_recent_trajectories[key], all_weight_matrices[key]):
            # 1. Isolate active neurons (using the final state of the window)
            final_state = recent_x[:, -1]
            active_mask = final_state > ACTIVE_THRESHOLD
            
            # 2. Record derivatives over the last 15 time steps
            # Vectorized calculation: -x + ReLU(W@x + 1)
            # W @ recent_x cleanly broadcasts to (N, 15)
            dx = -recent_x + np.maximum(0, W @ recent_x + 1)
            
            # 3. Calculate max absolute error of the derivative for each neuron
            max_dx = np.max(dx, axis=1)
            min_dx = np.min(dx, axis=1)
            diff_dx = np.abs(max_dx - min_dx)
            
            # Classify individual neurons
            fp_mask = diff_dx < DERIV_ERROR_THRESHOLD
            nl_mask = ~fp_mask
            
            # Filter for neurons that are BOTH active AND in the respective regime
            active_fp_count = np.sum(active_mask & fp_mask)
            active_nl_count = np.sum(active_mask & nl_mask)
            
            # 0. Classify the Realization (Trial)
            # If there is ANY active nonlinear neuron, the whole network is in a nonlinear regime
            if active_nl_count > 0:
                trial_nl_counts.append(active_nl_count)
            # If all active neurons are stable, it is a fixed point regime
            elif active_fp_count > 0:
                trial_fp_counts.append(active_fp_count)
                
        # 4. Take the average of the active neurons for each regime
        if len(trial_fp_counts) > 0:
            fp_mean_heatmap[q_idx, p_idx] = np.mean(trial_fp_counts)
        if len(trial_nl_counts) > 0:
            nl_mean_heatmap[q_idx, p_idx] = np.mean(trial_nl_counts)
            
        p_val = P_VALS[p_idx]
        q_val = Q_VALS[q_idx]
        
        # Helper strings to handle NaNs in printing
        fp_str = f"{np.mean(trial_fp_counts):.1f}" if len(trial_fp_counts) > 0 else "NaN"
        nl_str = f"{np.mean(trial_nl_counts):.1f}" if len(trial_nl_counts) > 0 else "NaN"
        
        print(f"p={p_val:.1f}, q={q_val:.1f} | Active FP Neurons: {fp_str} | Active NL Neurons: {nl_str}")

# ===================================================================
# 5. PLOTTING THE HEATMAPS
# ===================================================================

# Determine extent and padding for alignment
p_step = P_VALS[1] - P_VALS[0] if len(P_VALS) > 1 else 1
q_step = Q_VALS[1] - Q_VALS[0] if len(Q_VALS) > 1 else 1
extent = [
    P_VALS[0] - p_step/2, P_VALS[-1] + p_step/2, 
    Q_VALS[0] - q_step/2, Q_VALS[-1] + q_step/2
]

# Set up a colormap that highlights NaNs (empty regions) as light gray
import copy
cmap_fp = copy.copy(plt.get_cmap('cividis'))
cmap_fp.set_bad(color='whitesmoke')

cmap_nl = copy.copy(plt.get_cmap('plasma'))
cmap_nl.set_bad(color='whitesmoke')

# Global max for consistent color scaling
max_val = np.nanmax([np.nanmax(fp_mean_heatmap), np.nanmax(nl_mean_heatmap)])

fig_regimes, axes_regimes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
fig_regimes.suptitle(r'\Large Active Neurons by Dynamical Regime (N=' + f'{ANALYZE_MATRIX_SIZE})', 
                     fontsize=15, y=1.02)

# --- Overlay Curve Math ---
q_curve = np.linspace(0, 1, 100)
p_curve = (4 * q_curve) / (1 + q_curve)**2

# --- Panel A: Fixed Point Dynamics ---
im_fp = axes_regimes[0].imshow(fp_mean_heatmap, origin='lower', 
                               extent=extent, aspect='auto', 
                               cmap=cmap_fp, vmin=0, vmax=max_val)
axes_regimes[0].plot(p_curve, q_curve, color='white', linewidth=2, linestyle='--', 
                     label=r'$p = \frac{4q}{(1+q)^2}$')
axes_regimes[0].set_title(r'Fixed Point Realizations', fontsize=12)
axes_regimes[0].set_xlabel(r'$p$ (interaction probability)', fontsize=11)
axes_regimes[0].set_ylabel(r'$q$ (excitatory probability)', fontsize=11)
axes_regimes[0].set_xticks(P_VALS)
axes_regimes[0].set_yticks(Q_VALS)
axes_regimes[0].legend(loc='lower right')
cbar_fp = fig_regimes.colorbar(im_fp, ax=axes_regimes[0])
cbar_fp.set_label(r'Mean Active FP Neurons', fontsize=11)

# --- Panel B: Nonlinear Dynamics ---
im_nl = axes_regimes[1].imshow(nl_mean_heatmap, origin='lower', 
                               extent=extent, aspect='auto', 
                               cmap=cmap_nl, vmin=0, vmax=max_val)
axes_regimes[1].plot(p_curve, q_curve, color='white', linewidth=2, linestyle='--', 
                     label=r'$p = \frac{4q}{(1+q)^2}$')
axes_regimes[1].set_title(r'Nonlinear Realizations', fontsize=12)
axes_regimes[1].set_xlabel(r'$p$ (interaction probability)', fontsize=11)
axes_regimes[1].set_ylabel(r'$q$ (excitatory probability)', fontsize=11)
axes_regimes[1].set_xticks(P_VALS)
axes_regimes[1].set_yticks(Q_VALS)
axes_regimes[1].legend(loc='lower right')
cbar_nl = fig_regimes.colorbar(im_nl, ax=axes_regimes[1])
cbar_nl.set_label(r'Mean Active NL Neurons', fontsize=11)

# Save the figures
filepath_regimes_pdf = os.path.join(run_folder, 'dynamical_regimes_heatmap.pdf')
filepath_regimes_pgf = os.path.join(run_folder, 'dynamical_regimes_heatmap.pgf')
fig_regimes.savefig(filepath_regimes_pdf, bbox_inches='tight')
fig_regimes.savefig(filepath_regimes_pgf, bbox_inches='tight')

print(f"\nFigures saved to: {run_folder}/")
print(f"  - dynamical_regimes_heatmap.pdf/pgf")
print("="*70 + "\n")

BLOCK 2: DYNAMICAL REGIME CLASSIFICATION (FIXED POINT VS NONLINEAR)
Classifying network realizations...
  Active threshold: x > 0.05
  Fixed point threshold: max(dx) - min(dx) < 0.01 over last 15 steps

p=0.0, q=0.0 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=0.1 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=0.2 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=0.3 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=0.4 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=0.5 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=0.6 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=0.7 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=0.8 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=0.9 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.0, q=1.0 | Active FP Neurons: 1.0 | Active NL Neurons: NaN
p=0.1, q=0.0 | Active FP Neurons: 1.0 | Active NL Neurons: 2.0
p=0.1, q=0.1 | Active FP Neurons: 1.2 | A